In [ ]:
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════════╗
║         AgroIA Eventualidades v2.3                          ║
║         Evaluación satelital de daño agrícola               ║
║         con ponderación fenológica                          ║
║                                                              ║
║  Input:  KMZ | SHP | coordenadas manuales                   ║
║  Método: ΔNDVI relativo × peso fenológico × factor 0.92     ║
║  Fuente: Sentinel-2 L2A (Copernicus) + baseline 3 años      ║
╚════════════════════════════════════════════════════════════╝
"""
!pip install geemap geopandas -q

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import math, zipfile, os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
print('✅ Librerias listas')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.1 MB/s eta 0:00:00
✅ Librerias listas


In [ ]:
# ============================================================
# CELDA 2 — Autenticación GEE
# ============================================================
try:
    ee.Initialize(project='applied-oxygen-459415-e2')
    print('✅ GEE inicializado')
except:
    ee.Authenticate()
    ee.Initialize(project='applied-oxygen-459415-e2')
    print('✅ GEE autenticado e inicializado')

✅ GEE autenticado e inicializado


In [ ]:
# ============================================================
# CELDA 3 — TABLA FENOLOGICA + FACTOR CONSERVADOR
# ============================================================
TABLA_FENOLOGICA = {
    ('maiz',    1): ('Floracion / llenado de grano (R1-R4)',   1.0),
    ('maiz',    2): ('Llenado / madurez fisiologica (R4-R6)',  0.9),
    ('maiz',    3): ('Madurez / cosecha (R6+)',                0.2),
    ('maiz',    4): ('Post-cosecha / barbecho',                0.1),
    ('maiz',   11): ('Implantacion / V1-V3',                   0.3),
    ('maiz',   12): ('Vegetativo activo (V4-V10)',              0.6),
    ('soja',    1): ('Floracion / llenado (R1-R5)',             1.0),
    ('soja',    2): ('Llenado de grano / madurez (R5-R7)',      0.8),
    ('soja',    3): ('Madurez / cosecha (R8)',                  0.2),
    ('soja',   11): ('Implantacion / VC-V2',                   0.3),
    ('soja',   12): ('Vegetativo (V3-V6)',                     0.6),
    ('trigo',   7): ('Macollaje temprano (Z21-Z29)',            0.2),
    ('trigo',   8): ('Encañazon (Z30-Z39)',                    0.5),
    ('trigo',   9): ('Espigaon / antesis (Z55-Z69)',           1.0),
    ('trigo',  10): ('Antesis / llenado de grano (critico)',   1.0),
    ('trigo',  11): ('Madurez / cosecha',                      0.3),
    ('trigo',  12): ('Post-cosecha',                           0.1),
    ('cebada',  7): ('Macollaje temprano',                     0.2),
    ('cebada',  8): ('Encañazon',                              0.5),
    ('cebada',  9): ('Antesis (critico)',                      1.0),
    ('cebada', 10): ('Llenado de grano (critico)',             1.0),
    ('cebada', 11): ('Madurez / cosecha',                      0.3),
    ('girasol', 1): ('Floracion / llenado (R5-R7)',            1.0),
    ('girasol', 2): ('Madurez / cosecha (R8-R9)',              0.3),
    ('girasol',12): ('Vegetativo (V4-V10)',                    0.5),
}

FACTOR_CONSERVADOR = 0.92

def get_peso_fenologico(cultivo, mes):
    # Normalizar variantes comunes
    cultivo_norm = (cultivo.lower().strip()
                    .replace('maiz 2da','maiz').replace('maíz','maiz')
                    .replace('soja 1ra','soja').replace('soja 2da','soja')
                    .replace('maíz 2da','maiz'))
    key = (cultivo_norm, int(mes))
    if key in TABLA_FENOLOGICA:
        return TABLA_FENOLOGICA[key]
    print(f'  Combinacion {cultivo}/{mes} no en tabla. Usando peso neutro 0.5')
    return (f'Etapa no especificada ({cultivo}, mes {mes})', 0.5)

print('✅ Tabla fenologica cargada')
print(f'   Cultivos: maiz, soja, trigo, cebada, girasol')
print(f'   Factor conservador: {FACTOR_CONSERVADOR}')

✅ Tabla fenologica cargada
   Cultivos: maiz, soja, trigo, cebada, girasol
   Factor conservador: 0.92


In [ ]:
# ============================================================
# CELDA 4 — INGESTA DE POLIGONO
# ============================================================
# Elegir MODO y completar solo la sección correspondiente
#
#   'coordenadas' → pegar vértices [lon, lat] abajo
#   'kmz'         → subir .kmz a /content y poner nombre
#   'shp'         → subir .shp + .dbf + .shx a /content

MODO = 'coordenadas'   # <- CAMBIAR AQUI: 'coordenadas' | 'kmz' | 'shp'

# ── OPCION A: coordenadas manuales ──────────────────────────
# Vértices en orden [lon, lat] — cerrar repitiendo el primero
# Tip: desde Google Maps clic derecho → copiar coordenadas (lat, lon)
# y revertir el orden acá
COORDS = [
    [-61.9625089685879757, -32.7485209315490238],
    [-61.9595930381371218, -32.7485209315490238],
    [-61.9595930381371218, -32.7459331604736690],
    [-61.9625089685879757, -32.7459331604736690],
    [-61.9625089685879757, -32.7485209315490238]
]

# ── OPCION B: KMZ ───────────────────────────────────────────
KMZ_PATH = '/content/lote.kmz'   # <- nombre del archivo subido

# ── OPCION C: SHP ───────────────────────────────────────────
SHP_PATH = '/content/lote.shp'   # <- nombre del archivo subido

# ── Procesamiento automatico ─────────────────────────────────
if MODO == 'coordenadas':
    lote = ee.Geometry.Polygon([COORDS])
    print('✅ Poligono cargado desde coordenadas manuales')

elif MODO == 'kmz':
    with zipfile.ZipFile(KMZ_PATH, 'r') as z:
        z.extractall('/content/kmz_temp')
    kml_file = [f for f in os.listdir('/content/kmz_temp') if f.endswith('.kml')][0]
    gdf = gpd.read_file(f'/content/kmz_temp/{kml_file}').to_crs('EPSG:4326')
    geom = gdf.geometry.iloc[0]
    if geom.geom_type == 'MultiPolygon':
        geom = max(geom.geoms, key=lambda g: g.area)
    coords = [[lon, lat] for lon, lat in geom.exterior.coords]
    lote = ee.Geometry.Polygon([coords])
    print(f'✅ Poligono cargado desde KMZ: {KMZ_PATH}')

elif MODO == 'shp':
    gdf = gpd.read_file(SHP_PATH).to_crs('EPSG:4326')
    geom = gdf.geometry.iloc[0]
    if geom.geom_type == 'MultiPolygon':
        geom = max(geom.geoms, key=lambda g: g.area)
    coords = [[lon, lat] for lon, lat in geom.exterior.coords]
    lote = ee.Geometry.Polygon([coords])
    print(f'✅ Poligono cargado desde SHP: {SHP_PATH}')

area_preview = round(lote.area().divide(10000).getInfo(), 1)
print(f'   Superficie aproximada: {area_preview} ha')

✅ Poligono cargado desde coordenadas manuales
   Superficie aproximada: 7.8 ha


In [ ]:
# ============================================================
# CELDA 5 — CONFIGURACION DEL CASO  ← EDITAR AQUI
# ============================================================

CASO_NOMBRE  = 'Cordoba Marcos Juarez - Formación de grano'
FECHA_EVENTO = '2018-10-17'   # YYYY-MM-DD
CULTIVO      = 'trigo'         # maiz | soja | trigo | cebada | girasol
TIPO_EVENTO  = 'granizo'      # granizo | viento | inundacion | sequia

# ── Derivados automaticos ────────────────────────────────────
fecha_obj  = datetime.strptime(FECHA_EVENTO, '%Y-%m-%d')
mes_evento = fecha_obj.month

PRE_INI  = (fecha_obj - timedelta(days=14)).strftime('%Y-%m-%d')
PRE_FIN  = (fecha_obj - timedelta(days=1)).strftime('%Y-%m-%d')
POST_INI = (fecha_obj + timedelta(days=1)).strftime('%Y-%m-%d')
POST_FIN = (fecha_obj + timedelta(days=20)).strftime('%Y-%m-%d')

etapa_desc, peso_fenologico = get_peso_fenologico(CULTIVO, mes_evento)

print('=' * 62)
print(f'Caso:    {CASO_NOMBRE}')
print(f'Evento:  {FECHA_EVENTO} ({TIPO_EVENTO})')
print(f'Cultivo: {CULTIVO.upper()} — {etapa_desc}')
print(f'Peso fenologico: {peso_fenologico}  |  Factor conservador: {FACTOR_CONSERVADOR}')
print(f'Ventana PRE:  {PRE_INI} → {PRE_FIN}')
print(f'Ventana POST: {POST_INI} → {POST_FIN}')
print('=' * 62)

Caso:    Cordoba Marcos Juarez - Formación de grano
Evento:  2018-10-17 (granizo)
Cultivo: TRIGO — Antesis / llenado de grano (critico)
Peso fenologico: 1.0  |  Factor conservador: 0.92
Ventana PRE:  2018-10-03 → 2018-10-16
Ventana POST: 2018-10-18 → 2018-11-06


In [ ]:
# ============================================================
# CELDA 6 — Funciones de procesamiento Sentinel-2
# ============================================================

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = (qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0)))
    return image.updateMask(mask).divide(10000)

def add_ndvi(image):
    return image.addBands(image.normalizedDifference(['B8','B4']).rename('NDVI'))

def get_median_ndvi(start_date, end_date, geometry):
    """NDVI mediano con fallback SR→TOA para fechas historicas (<2019)."""
    for col_name in ['COPERNICUS/S2_SR_HARMONIZED', 'COPERNICUS/S2_HARMONIZED']:
        col = (ee.ImageCollection(col_name)
               .filterBounds(geometry).filterDate(start_date, end_date)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35))
               .map(mask_s2_clouds).map(add_ndvi).select('NDVI'))
        count = col.size().getInfo()
        if count > 0:
            print(f'  {start_date}→{end_date}: {count} img [{col_name.split("/")[1]}]')
            return col.median().clip(geometry), count
    print(f'  Sin imagenes limpias. Relajando umbral de nubes...')
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(geometry).filterDate(start_date, end_date)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
           .map(mask_s2_clouds).map(add_ndvi).select('NDVI'))
    count = col.size().getInfo()
    print(f'  {count} img (filtro relajado)')
    # Return a single-band image with 0 if no images are found, to prevent band mismatch errors.
    if count == 0:
        return ee.Image(0).rename('NDVI'), 0
    return col.median().clip(geometry), count

def calculate_area(mask, geometry, scale=10):
    result = (mask.rename('m').multiply(ee.Image.pixelArea())
              .reduceRegion(ee.Reducer.sum(), geometry, scale, maxPixels=1e9, bestEffort=True))
    return result.getNumber('m').getInfo() / 10000

def get_mean(img, band, geom=None, scale=10):
    g = geom if geom else lote
    return round(img.reduceRegion(ee.Reducer.mean(), g, scale, bestEffort=True).getInfo().get(band, 0), 3)

print('✅ Funciones listas')


✅ Funciones listas


In [ ]:
# ============================================================
# CELDA 7 — NDVI pre y post evento
# ============================================================
print('Descargando Sentinel-2...')
print('PRE-evento:')
ndvi_pre,  n_pre  = get_median_ndvi(PRE_INI,  PRE_FIN,  lote)
print('POST-evento:')
ndvi_post, n_post = get_median_ndvi(POST_INI, POST_FIN, lote)

confianza = 'ALTA' if n_pre >= 3 and n_post >= 3 else 'MEDIA' if n_pre >= 1 and n_post >= 1 else 'BAJA'
print(f'\n✅ Confianza: {confianza} ({n_pre} img PRE / {n_post} img POST)')

Descargando Sentinel-2...
PRE-evento:
  2018-10-03→2018-10-16: 1 img [S2_SR_HARMONIZED]
POST-evento:
  2018-10-18→2018-11-06: 3 img [S2_HARMONIZED]

✅ Confianza: MEDIA (1 img PRE / 3 img POST)


In [ ]:
# ============================================================
# CELDA 8 — Baseline historico 3 años
# ============================================================
anios = [fecha_obj.year - i for i in range(1, 4)]
print(f'Calculando baseline: {anios}...')

def get_baseline_year(year, geometry):
    try:
        pre,  _ = get_median_ndvi(
            (fecha_obj.replace(year=year) - timedelta(days=14)).strftime('%Y-%m-%d'),
            (fecha_obj.replace(year=year) - timedelta(days=1)).strftime('%Y-%m-%d'),
            geometry)
        post, _ = get_median_ndvi(
            (fecha_obj.replace(year=year) + timedelta(days=1)).strftime('%Y-%m-%d'),
            (fecha_obj.replace(year=year) + timedelta(days=20)).strftime('%Y-%m-%d'),
            geometry)
        # Ensure the output has a consistent band name for averaging later.
        return post.subtract(pre).rename('delta_year')
    except Exception as e:
        print(f'  Error anio {year}: {e}. Usando imagen neutra.')
        # Ensure the fallback image also has the consistent band name.
        return ee.Image(0).rename('delta_year')

deltas = [get_baseline_year(y, lote) for y in anios]
# Use ee.ImageCollection.mean() for robust averaging of images with consistent bands.
baseline_3y = ee.ImageCollection(deltas).mean().rename('baseline_3y')
print('✅ Baseline calculado')


Calculando baseline: [2017, 2016, 2015]...
  2017-10-03→2017-10-16: 1 img [S2_SR_HARMONIZED]
  2017-10-18→2017-11-06: 3 img [S2_HARMONIZED]
  2016-10-03→2016-10-16: 4 img [S2_HARMONIZED]
  2016-10-18→2016-11-06: 1 img [S2_SR_HARMONIZED]
  Sin imagenes limpias. Relajando umbral de nubes...
  0 img (filtro relajado)
  Sin imagenes limpias. Relajando umbral de nubes...
  0 img (filtro relajado)
✅ Baseline calculado


In [ ]:
# ============================================================
# CELDA 9 — CALCULO DE DAÑO PONDERADO
# ============================================================
#
#   delta_rel      = (NDVI_pre - NDVI_post) / NDVI_pre x 100
#   dano_ponderado = delta_rel x peso_fenologico x factor_conservador
#
#   Clasificacion espacial por pixel:
#   Leve     20-40%  ponderado
#   Moderado 40-70%  ponderado
#   Severo   >70%    ponderado

delta_obs = ndvi_post.subtract(ndvi_pre).rename('delta_observado')
delta_adj = delta_obs.subtract(baseline_3y).rename('delta_ajustado')

epsilon   = ee.Image(0.01)
delta_rel = (ndvi_pre.subtract(ndvi_post)
             .divide(ndvi_pre.max(epsilon))
             .multiply(100).max(ee.Image(0))
             .rename('delta_relativo'))

dano_pond_img = (delta_rel
                 .multiply(peso_fenologico)
                 .multiply(FACTOR_CONSERVADOR)
                 .rename('dano_ponderado'))

mask_leve     = dano_pond_img.gte(20).And(dano_pond_img.lt(40))
mask_moderado = dano_pond_img.gte(40).And(dano_pond_img.lt(70))
mask_severo   = dano_pond_img.gte(70)

severidad = (ee.Image.cat([
    mask_leve.multiply(1),
    mask_moderado.multiply(2),
    mask_severo.multiply(3)
]).reduce(ee.Reducer.max()).rename('severidad'))

print(f'✅ Daño ponderado calculado')
print(f'   Peso fenologico: {peso_fenologico} | Factor conservador: {FACTOR_CONSERVADOR}')

✅ Daño ponderado calculado
   Peso fenologico: 1.0 | Factor conservador: 0.92


In [ ]:
# ============================================================
# CELDA 10 — Estadisticas
# ============================================================
area_total    = round(lote.area().divide(10000).getInfo(), 1)
ndvi_pre_val  = get_mean(ndvi_pre,    'NDVI')
ndvi_post_val = get_mean(ndvi_post,   'NDVI')
baseline_val  = get_mean(baseline_3y, 'baseline_3y')
delta_adj_val = get_mean(delta_adj,   'delta_ajustado')

delta_rel_val = round(max((ndvi_pre_val - ndvi_post_val) / max(ndvi_pre_val, 0.01) * 100, 0), 1)
dano_pond_val = round(delta_rel_val * peso_fenologico * FACTOR_CONSERVADOR, 1)

area_leve     = calculate_area(severidad.eq(1), lote)
area_moderada = calculate_area(severidad.eq(2), lote)
area_severa   = calculate_area(severidad.eq(3), lote)
area_afectada = area_leve + area_moderada + area_severa

clasificacion = 'LEVE' if dano_pond_val < 20 else 'MODERADO' if dano_pond_val < 40 else 'SEVERO'

sep = '=' * 68
lin = '-' * 68
print(sep)
print('        INFORME AgroIA Eventualidades v2.1')
print(sep)
print(f'Caso:    {CASO_NOMBRE}')
print(f'Evento:  {FECHA_EVENTO} ({TIPO_EVENTO})')
print(f'Cultivo: {CULTIVO.upper()} — {etapa_desc}')
print(f'Confianza: {confianza} ({n_pre} img PRE / {n_post} img POST)')
print(lin)
print('METRICAS DE VEGETACION')
print(lin)
print(f'  NDVI pre:          {ndvi_pre_val}')
print(f'  NDVI post:         {ndvi_post_val}')
print(f'  Delta relativo:    {delta_rel_val}%')
print(f'  Baseline (3a):     {baseline_val}')
print(f'  Delta ajustado:    {delta_adj_val}')
print(lin)
print('PONDERACION FENOLOGICA')
print(lin)
print(f'  {delta_rel_val}% x {peso_fenologico} (peso) x {FACTOR_CONSERVADOR} (factor) = {dano_pond_val}% → {clasificacion}')
print(lin)
print('SUPERFICIE')
print(lin)
print(f'  Total:             {area_total} ha')
print(f'  Leve   (20-40%):   {area_leve:.1f} ha  ({area_leve/area_total*100:.1f}%)')
print(f'  Moderado(40-70%):  {area_moderada:.1f} ha  ({area_moderada/area_total*100:.1f}%)')
print(f'  Severo (>70%):     {area_severa:.1f} ha  ({area_severa/area_total*100:.1f}%)')
print(f'  TOTAL AFECTADO:    {area_afectada:.1f} ha  ({area_afectada/area_total*100:.1f}%)')
print(sep)

        INFORME AgroIA Eventualidades v2.1
Caso:    Cordoba Marcos Juarez - Formación de grano
Evento:  2018-10-17 (granizo)
Cultivo: TRIGO — Antesis / llenado de grano (critico)
Confianza: MEDIA (1 img PRE / 3 img POST)
--------------------------------------------------------------------
METRICAS DE VEGETACION
--------------------------------------------------------------------
  NDVI pre:          0.556
  NDVI post:         0.328
  Delta relativo:    41.0%
  Baseline (3a):     0.061
  Delta ajustado:    -0.289
--------------------------------------------------------------------
PONDERACION FENOLOGICA
--------------------------------------------------------------------
  41.0% x 1.0 (peso) x 0.92 (factor) = 37.7% → MODERADO
--------------------------------------------------------------------
SUPERFICIE
--------------------------------------------------------------------
  Total:             7.8 ha
  Leve   (20-40%):   3.0 ha  (37.9%)
  Moderado(40-70%):  3.8 ha  (48.2%)
  Severo (>70%

In [ ]:
# ============================================================
# CELDA 11 — MAPA INTERACTIVO + PUNTOS DE MUESTREO
# ============================================================
# Fondo: Google Satellite Hybrid
# Capas: NDVI pre/post, daño ponderado, severidad
# Puntos: 5 por categoria (Leve/Moderado/Severo) para peritaje
# Extras: boton GPS, medicion de distancias, KML exportable

!pip install folium simplekml -q
import folium
from folium import plugins
import simplekml

# ── 1. Puntos de muestreo estratificados ─────────────────────
print('Generando puntos de muestreo...')

def generar_puntos_muestreo(raster_severidad, n_puntos=5):
    puntos = []
    for cat, nombre in zip([1, 2, 3], ['Leve', 'Moderado', 'Severo']):
        mask = raster_severidad.eq(cat)
        muestras = mask.stratifiedSample(
            numPoints=n_puntos, region=lote,
            scale=10, geometries=True
        ).getInfo()
        for feat in muestras['features']:
            lon, lat = feat['geometry']['coordinates']
            puntos.append({
                'Categoria': nombre,
                'Latitud':   round(lat, 6),
                'Longitud':  round(lon, 6),
                'Google_Maps': f'https://www.google.com/maps?q={lat:.6f},{lon:.6f}'
            })
    return pd.DataFrame(puntos)

df_muestreo = generar_puntos_muestreo(severidad, n_puntos=5)
print(f'✅ {len(df_muestreo)} puntos generados')
print(df_muestreo[['Categoria','Latitud','Longitud']].to_string(index=False))

# ── 2. URLs de capas GEE ─────────────────────────────────────
def get_tile_url(img, vis):
    return ee.Image(img).getMapId(vis)['tile_fetcher'].url_format

url_pre  = get_tile_url(ndvi_pre,  {'min':0,'max':0.8,'palette':['#8B4513','#DAA520','#228B22','#006400']})
url_post = get_tile_url(ndvi_post, {'min':0,'max':0.8,'palette':['#8B4513','#DAA520','#228B22','#006400']})
url_dano = get_tile_url(dano_pond_img, {'min':0,'max':100,'palette':['#F0F0F0','#FFD700','#FF8C00','#8B0000']})
url_sev  = get_tile_url(severidad.selfMask(), {'min':1,'max':3,'palette':['#FFD700','#FF8C00','#8B0000']})

# ── 3. Mapa Folium con Google Hybrid ─────────────────────────
centro = lote.centroid().coordinates().getInfo()[::-1]

m = folium.Map(
    location=centro, zoom_start=13,
    control_scale=True,
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google Satellite Hybrid'
)

# Capas GEE
for url, name, opacity in [
    (url_pre,  'NDVI Pre-evento',       0.75),
    (url_post, 'NDVI Post-evento',      0.75),
    (url_dano, f'Daño ponderado % (peso={peso_fenologico})', 0.80),
    (url_sev,  'Severidad del daño',    0.85),
]:
    folium.TileLayer(
        tiles=url, attr='GEE — AgroIA',
        name=name, overlay=True,
        control=True, opacity=opacity,
        show=(name == 'Severidad del daño')
    ).add_to(m)

# Borde del lote
lote_coords = lote.coordinates().getInfo()[0]
folium.Polygon(
    locations=[[lat, lon] for lon, lat in lote_coords],
    color='black', weight=3, fill=False,
    tooltip='Limite del lote'
).add_to(m)

# Marcadores por categoria
colores = {'Leve': 'cadetblue', 'Moderado': 'orange', 'Severo': 'red'}
iconos  = {'Leve': 'info-sign', 'Moderado': 'warning-sign', 'Severo': 'exclamation-sign'}

for _, row in df_muestreo.iterrows():
    cat = row['Categoria']
    folium.Marker(
        location=[row['Latitud'], row['Longitud']],
        popup=folium.Popup(
            f"<b>Categoria: {cat}</b><br>"
            f"Lat: {row['Latitud']}<br>Lon: {row['Longitud']}<br>"
            f"<a href='{row['Google_Maps']}' target='_blank'>Ver en Google Maps</a>",
            max_width=220),
        tooltip=f'Punto {cat}',
        icon=folium.Icon(color=colores[cat], icon=iconos[cat])
    ).add_to(m)

# Herramientas de campo
plugins.LocateControl(auto_start=False).add_to(m)       # GPS
plugins.MeasureControl(primary_length_unit='meters').add_to(m)  # Medicion
folium.LayerControl(collapsed=False).add_to(m)

# ── 4. Guardar HTML del visor ─────────────────────────────────
visor_nombre = f'visor_agroia_{CULTIVO}_{FECHA_EVENTO}.html'
m.save(visor_nombre)
print(f'\n✅ Visor guardado: {visor_nombre}')

# ── 5. Exportar KML para GPS / Google Earth ───────────────────
kml = simplekml.Kml(name=f'AgroIA — {CASO_NOMBRE}')
estilos_kml = {'Severo': 'ff0000ff', 'Moderado': 'ff0064ff', 'Leve': 'ff00ffff'}

for _, row in df_muestreo.iterrows():
    cat = row['Categoria']
    pnt = kml.newpoint(
        name=cat,
        coords=[(row['Longitud'], row['Latitud'])]
    )
    pnt.style.iconstyle.color = estilos_kml.get(cat, 'ffffffff')
    pnt.style.iconstyle.scale = 1.2
    pnt.description = (
        f'Punto de validacion AgroIA\n'
        f'Categoria: {cat}\n'
        f'Caso: {CASO_NOMBRE}\n'
        f'Fecha: {FECHA_EVENTO}'
    )

kml_nombre = f'puntos_muestreo_{CULTIVO}_{FECHA_EVENTO}.kml'
kml.save(kml_nombre)
print(f'✅ KML guardado: {kml_nombre}')

# ── 6. CSV para perito ────────────────────────────────────────
csv_nombre = f'puntos_muestreo_{CULTIVO}_{FECHA_EVENTO}.csv'
df_muestreo.to_csv(csv_nombre, index=False)
print(f'✅ CSV guardado: {csv_nombre}')

m  # Mostrar mapa en Colab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Generando puntos de muestreo...
✅ 25 puntos generados
Categoria    Latitud   Longitud
     Leve -32.747140 -61.961970
     Leve -32.746512 -61.961431
     Leve -32.747590 -61.959635
     Leve -32.747140 -61.959725
     Leve -32.747949 -61.961970
     Leve -32.746781 -61.962509
     Leve -32.748308 -61.961701
     Leve -32.746422 -61.959994
     Leve -32.747500 -61.960803
     Leve -32.747140 -61.962330
 Moderado -32.746781 -61.962509
 Moderado -32.747140 -61.959725
 Moderado -32.748308 -61.961701
 Moderado -32.748398 -61.961970
 Moderado -32.746422 -61.959994
 Moderado -32.747140 -61.961970
 Moderado -32.746512 -61.961431
 Moderado -32.747590 -61.959635
 Moderado -32.747949 -61.961970
 Moderado -32.746781 -61.960264
   Severo -32.747140 -61.961970
   Severo -32.746512 -61.961431
   Severo -32.746781 -61.962509
   Severo -32.747590 -61.959635
   Severo -32.747140 -61.

In [ ]:
import pytz
from datetime import datetime

# Get current time in Argentina/Buenos_Aires timezone
bas_tz = pytz.timezone('America/Argentina/Buenos_Aires')
time_in_bas = datetime.now(bas_tz).strftime('%d/%m/%Y %H:%M')

# Modify the HTML content to use time_in_bas
nombre_html = f'reporte_agroia_{CULTIVO}_{FECHA_EVENTO}.html'

html = f"""
<!DOCTYPE html><html lang='es'><head><meta charset='UTF-8'>
<title>AgroIA Eventualidades — {CASO_NOMBRE}</title>
<style>
body{{font-family:'Segoe UI',Arial,sans-serif;margin:0;padding:20px;background:#f0f4f8;}}
.wrap{{max-width:960px;margin:auto;background:#fff;border-radius:16px;padding:32px;box-shadow:0 8px 30px rgba(0,0,0,.1);}}
h1{{color:#1a1a2e;border-left:6px solid #8B0000;padding-left:16px;font-size:1.4em;}}
h2{{color:#333;margin-top:28px;}}
.badge{{display:inline-block;padding:6px 18px;border-radius:20px;font-weight:bold;font-size:1.1em;}}
.leve{{background:#FFD700;color:#333;}}.moderado{{background:#FF8C00;color:#fff;}}.severo{{background:#8B0000;color:#fff;}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:14px;margin:20px 0;}}
.card{{background:#f7f9fc;border-radius:12px;padding:16px;text-align:center;}}
.num{{font-size:1.9em;font-weight:bold;color:#8B0000;}}
.formula{{background:#f0f4f8;border-left:4px solid #8B0000;padding:14px;border-radius:8px;font-family:monospace;margin:16px 0;}}
table{{width:100%;border-collapse:collapse;margin:12px 0;}}
th{{background:#8B0000;color:#fff;padding:10px;text-align:left;}}
td{{padding:9px;border-bottom:1px solid #eee;}}
tr:nth-child(even){{background:#fafafa;}}
.conf-alta{{color:#2e7d32;font-weight:bold;}}.conf-media{{color:#f57c00;font-weight:bold;}}.conf-baja{{color:#c62828;font-weight:bold;}}
.footer{{text-align:center;margin-top:28px;padding-top:18px;border-top:1px solid #ddd;color:#888;font-size:.85em;}}
</style></head><body><div class='wrap'>
<h1>Informe de Eventualidad Agricola — AgroIA</h1>
<p><b>Caso:</b> {CASO_NOMBRE} &nbsp;|&nbsp; <b>Evento:</b> {FECHA_EVENTO} ({TIPO_EVENTO}) &nbsp;|&nbsp; <b>Cultivo:</b> {CULTIVO.upper()}</p>
<p><b>Etapa fenologica:</b> {etapa_desc}</p>
<p><b>Confianza del analisis:</b> <span class='conf-{confianza.lower()}'>{confianza}</span> — {n_pre} imagenes PRE / {n_post} imagenes POST</p>
<h2>Resultado</h2>
<p>Daño ponderado estimado: <span class='badge {clasificacion.lower()}'>{dano_pond_val}% — {clasificacion}</span></p>
<div class='grid'>
  <div class='card'><div>Area total</div><div class='num'>{area_total} ha</div></div>
  <div class='card'><div>Area afectada</div><div class='num'>{area_afectada:.0f} ha</div><div>({area_afectada/area_total*100:.1f}%)</div></div>
  <div class='card'><div>NDVI pre</div><div class='num'>{ndvi_pre_val}</div></div>
  <div class='card'><div>NDVI post</div><div class='num'>{ndvi_post_val}</div></div>
  <div class='card'><div>Delta relativo</div><div class='num'>{delta_rel_val}%</div></div>
  <div class='card'><div>Daño ponderado</div><div class='num'>{dano_pond_val}%</div></div>
</div>
<h2>Metodologia</h2>
<div class='formula'>
delta_rel      = (NDVI_pre − NDVI_post) / NDVI_pre × 100 = {delta_rel_val}%<br>
dano_ponderado = {delta_rel_val}% × {peso_fenologico} (peso fenologico: {etapa_desc}) × {FACTOR_CONSERVADOR} (factor conservador)<br>
               = <b>{dano_pond_val}%</b>
</div>
<h2>Distribucion espacial del daño</h2>
<table>
<tr><th>Categoria</th><th>Area (ha)</th><th>% del lote</th></tr>
<tr><td>🟡 Leve (20-40%)</td><td>{area_leve:.1f}</td><td>{area_leve/area_total*100:.1f}%</td></tr>
<tr><td>🟠 Moderado (40-70%)</td><td>{area_moderada:.1f}</td><td>{area_moderada/area_total*100:.1f}%</td></tr>
<tr><td>🔴 Severo (>70%)</td><td>{area_severa:.1f}</td><td>{area_severa/area_total*100:.1f}%</td></tr>
<tr><td><b>Total afectado</b></td><td><b>{area_afectada:.1f}</b></td><td><b>{area_afectada/area_total*100:.1f}%</b></td></tr>
</table>
<h2>Trazabilidad</h2>
<table>
<tr><th>Parametro</th><th>Detalle</th></tr>
<tr><td>Sensor</td><td>Sentinel-2 MSI — Copernicus / ESA</td></tr>
<tr><td>Resolucion</td><td>10 m/pixel</td></tr>
<tr><td>Indice</td><td>NDVI = (B8 − B4) / (B8 + B4)</td></tr>
<tr><td>Baseline</td><td>{anios[2]}–{anios[0]} (misma ventana fenologica)</td></tr>
<tr><td>Confianza</td><td>{confianza} — {n_pre} img PRE / {n_post} img POST</td></tr>
<tr><td>Procesamiento</td><td>Google Earth Engine — AgroIA Eventualidades v2.3</td></tr>
<tr><td>Generado</td><td>{time_in_bas}</td>
</tr>
</table>
<div class='footer'>
<p>AgroIA Eventualidades v2.3 | Sentinel-2 L2A (Copernicus) | Baseline 3 años</p>
<p>Los valores son estimaciones satelitales. Se recomienda validacion en campo para siniestros formales.</p>
</div></div></body></html>
"""

with open(nombre_html, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'✅ Reporte HTML: {nombre_html}')


✅ Reporte HTML: reporte_agroia_trigo_2018-10-17.html


In [ ]:
# ============================================================
# CELDA 13 — Exportar capas a Google Drive
# ============================================================
carpeta = f'AgroIA_{CULTIVO}_{FECHA_EVENTO}'

exports = [
    (ndvi_pre,      'ndvi_pre',       {'min':0,'max':0.8,'palette':['#8B3A3A','#D4A017','#4CAF50','#1B5E20']}),
    (ndvi_post,     'ndvi_post',      {'min':0,'max':0.8,'palette':['#8B3A3A','#D4A017','#4CAF50','#1B5E20']}),
    (delta_adj,     'delta_ajustado', {'min':-0.5,'max':0.1,'palette':['#8B0000','#FF4500','#FFD700','#F0F0F0']}),
    (dano_pond_img, 'dano_ponderado', {'min':0,'max':100,'palette':['#F0F0F0','#FFD700','#FF8C00','#8B0000']}),
    (severidad,     'mapa_severidad', {'min':0,'max':3,'palette':['#FFFFFF','#FFD700','#FF8C00','#8B0000']}),
]

for img, name, vis in exports:
    task = ee.batch.Export.image.toDrive(
        image=img.visualize(**vis), description=name,
        folder=carpeta, fileNamePrefix=name,
        region=lote, scale=10, crs='EPSG:4326', maxPixels=1e9)
    task.start()
    print(f'  Exportando: {name} → Drive/{carpeta}')

print()
print('✅ PIPELINE COMPLETADO')
print(f'   Caso:           {CASO_NOMBRE}')
print(f'   Cultivo:        {CULTIVO.upper()} — {etapa_desc}')
print(f'   Daño ponderado: {dano_pond_val}% ({clasificacion})')
print(f'   Area afectada:  {area_afectada:.1f} ha de {area_total} ha')
print(f'   Confianza:      {confianza} ({n_pre} img PRE / {n_post} img POST)')
print(f'   Reporte HTML:   {nombre_html}')

  Exportando: ndvi_pre → Drive/AgroIA_trigo_2018-10-17
  Exportando: ndvi_post → Drive/AgroIA_trigo_2018-10-17
  Exportando: delta_ajustado → Drive/AgroIA_trigo_2018-10-17
  Exportando: dano_ponderado → Drive/AgroIA_trigo_2018-10-17
  Exportando: mapa_severidad → Drive/AgroIA_trigo_2018-10-17

✅ PIPELINE COMPLETADO
   Caso:           Cordoba Marcos Juarez - Formación de grano
   Cultivo:        TRIGO — Antesis / llenado de grano (critico)
   Daño ponderado: 37.7% (MODERADO)
   Area afectada:  6.7 ha de 7.8 ha
   Confianza:      MEDIA (1 img PRE / 3 img POST)
   Reporte HTML:   reporte_agroia_trigo_2018-10-17.html
